# MobileNetV3 Transfer Learning Experiments

## Objective

The objective of this notebook is to investigate the performance of MobileNetV3 Small for the screen recapture detection task.

MobileNetV3 is a lightweight convolutional neural network designed for efficient inference on resource-constrained devices. The model is fine-tuned using transfer learning and evaluated using Stratified 5-Fold Cross Validation.

The experiment serves as one of several deep learning baselines that are later compared with ResNet18, EfficientNet-B0, and ResNet34.

## Import Required Libraries

Import all libraries required for dataset loading, transfer learning, optimization, model evaluation, and cross-validation.

In [15]:
import os
import copy
import numpy as np

import torch
import torch.nn as nn
import torch.optim as optim

from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader, Subset

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

## Image Preprocessing

Define separate preprocessing pipelines for training and validation.

Training images are augmented to improve generalization, while validation images undergo deterministic preprocessing.

In [16]:
train_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomCrop(224),

    transforms.RandomHorizontalFlip(p=0.5),

    transforms.ColorJitter(
        brightness=0.1,
        contrast=0.1
    ),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=[0.485,0.456,0.406],
        std=[0.229,0.224,0.225]
    )
])



test_transform = transforms.Compose([

    transforms.Resize((224,224)),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=[0.485,0.456,0.406],
        std=[0.229,0.224,0.225]
    )

])

## Load Dataset

Load the dataset using the ImageFolder API and retrieve the corresponding class labels for cross-validation.

In [17]:
dataset_path = "../dataset"

dataset = datasets.ImageFolder(dataset_path)

labels = dataset.targets

In [18]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(device)

cpu


## Define Training and Evaluation Functions

Implement reusable functions for one training epoch and model evaluation.

These helper functions simplify the cross-validation training loop.

In [19]:
def train_one_epoch(model, loader):

    model.train()

    running_loss = 0
    correct = 0
    total = 0

    for images, labels in loader:

        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)

        loss = criterion(outputs, labels)

        loss.backward()

        optimizer.step()

        running_loss += loss.item()

        _, preds = torch.max(outputs, 1)

        total += labels.size(0)

        correct += (preds == labels).sum().item()

    epoch_loss = running_loss / len(loader)
    epoch_acc = correct / total

    return epoch_loss, epoch_acc

In [20]:
def evaluate(model, loader):

    model.eval()

    predictions = []
    targets = []

    correct = 0
    total = 0

    with torch.no_grad():

        for images, labels in loader:

            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)

            _, preds = torch.max(outputs, 1)

            predictions.extend(preds.cpu().numpy())
            targets.extend(labels.cpu().numpy())

            total += labels.size(0)
            correct += (preds == labels).sum().item()

    accuracy = correct / total

    return accuracy, predictions, targets

## Configure Stratified 5-Fold Cross Validation

Split the dataset into five stratified folds while preserving class balance in every validation set.

In [21]:
skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

## Train MobileNetV3

Fine-tune a pretrained MobileNetV3 Small model using transfer learning.

The experiment includes:

- Frozen backbone
- Fine-tuned classifier
- AdamW optimizer
- Label smoothing
- Learning-rate scheduling
- Early stopping

In [ ]:
fold_results = []

best_fold_accuracy = 0
best_fold = -1

os.makedirs("../models", exist_ok=True)

EPOCHS = 40
PATIENCE = 5

for fold, (train_idx, val_idx) in enumerate(skf.split(np.arange(len(dataset)), labels), 1):

    print("=" * 60)
    print(f"Fold {fold}")
    print("=" * 60)

    train_dataset = copy.deepcopy(dataset)
    train_dataset.transform = train_transform

    val_dataset = copy.deepcopy(dataset)
    val_dataset.transform = test_transform

    train_subset = Subset(train_dataset, train_idx)
    val_subset = Subset(val_dataset, val_idx)

    train_loader = DataLoader(
        train_subset,
        batch_size=8,
        shuffle=True
    )

    val_loader = DataLoader(
        val_subset,
        batch_size=8,
        shuffle=False
    )

    # ---------------- Model ---------------- #

    weights = models.MobileNet_V3_Small_Weights.DEFAULT

    model = models.mobilenet_v3_small(weights=weights)

    # Freeze entire backbone
    for param in model.features.parameters():
        param.requires_grad = False

    # Fine tune last FOUR blocks
    for param in model.features[-4:].parameters():
        param.requires_grad = True

    # Replace classifier
    model.classifier[3] = nn.Linear(1024, 2)

    model = model.to(device)

    # ---------------- Loss ---------------- #

    criterion = nn.CrossEntropyLoss(
        label_smoothing=0.1
    )

    # ---------------- Optimizer ---------------- #

    optimizer = optim.AdamW(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=5e-5,
        weight_decay=1e-4
    )

    # ---------------- Scheduler ---------------- #

    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode="max",
        factor=0.5,
        patience=3
    )

    # ---------------- Early Stopping ---------------- #

    best_acc = 0

    counter = 0

    best_model = copy.deepcopy(model.state_dict())

    # ---------------- Training ---------------- #

    for epoch in range(EPOCHS):

        loss, train_acc = train_one_epoch(model, train_loader)

        val_acc, preds, targets = evaluate(model, val_loader)

        scheduler.step(val_acc)

        print(
            f"Epoch {epoch+1:02d} | "
            f"Loss: {loss:.4f} | "
            f"Train: {train_acc:.4f} | "
            f"Val: {val_acc:.4f}"
        )

        if val_acc > best_acc:

            best_acc = val_acc
            best_model = copy.deepcopy(model.state_dict())
            counter = 0

        else:

            counter += 1

        if counter >= PATIENCE:

            print("Early Stopping...")
            break

    # ---------------- Load Best Model ---------------- #

    model.load_state_dict(best_model)

    val_acc, preds, targets = evaluate(model, val_loader)

    precision = precision_score(targets, preds)
    recall = recall_score(targets, preds)
    f1 = f1_score(targets, preds)

    print(f"\nBest Validation Accuracy : {val_acc:.4f}")

    # ---------------- Save Best Fold ---------------- #

    if val_acc > best_fold_accuracy:

        best_fold_accuracy = val_acc
        best_fold = fold

        torch.save(
            model.state_dict(),
            "../models/best_mobilenetv3_real_detector.pth"
        )

        print("Best model saved!")

    fold_results.append({

        "accuracy": val_acc,
        "precision": precision,
        "recall": recall,
        "f1": f1

    })



Fold 1
Epoch 01 | Loss: 0.6955 | Train: 0.5309 | Val: 0.5714
Epoch 02 | Loss: 0.6338 | Train: 0.6543 | Val: 0.5238
Epoch 03 | Loss: 0.6268 | Train: 0.7284 | Val: 0.5238
Epoch 04 | Loss: 0.6320 | Train: 0.6790 | Val: 0.7143
Epoch 05 | Loss: 0.5431 | Train: 0.8025 | Val: 0.7143
Epoch 06 | Loss: 0.5401 | Train: 0.8272 | Val: 0.7143
Epoch 07 | Loss: 0.5182 | Train: 0.9012 | Val: 0.7143
Epoch 08 | Loss: 0.5055 | Train: 0.8889 | Val: 0.7619
Epoch 09 | Loss: 0.4783 | Train: 0.9136 | Val: 0.7143
Epoch 10 | Loss: 0.4623 | Train: 0.9259 | Val: 0.7143
Epoch 11 | Loss: 0.4367 | Train: 0.9259 | Val: 0.7143
Epoch 12 | Loss: 0.4216 | Train: 0.9506 | Val: 0.7619
Epoch 13 | Loss: 0.4057 | Train: 0.9259 | Val: 0.7619
Early Stopping...

Best Validation Accuracy : 0.7619
Best model saved!
Fold 2
Epoch 01 | Loss: 0.7252 | Train: 0.4444 | Val: 0.6190
Epoch 02 | Loss: 0.6748 | Train: 0.5432 | Val: 0.6667
Epoch 03 | Loss: 0.6525 | Train: 0.6173 | Val: 0.6190
Epoch 04 | Loss: 0.6066 | Train: 0.7778 | Val: 0.61

## Evaluate Cross-Validation Performance

Calculate Accuracy, Precision, Recall, and F1 Score for every fold and report the mean performance across the entire experiment.

In [23]:
# ---------------- Results ---------------- #

acc = [x["accuracy"] for x in fold_results]
prec = [x["precision"] for x in fold_results]
rec = [x["recall"] for x in fold_results]
f1 = [x["f1"] for x in fold_results]

print("\n" + "=" * 60)
print("Cross Validation Results")
print("=" * 60)

print("Accuracy :", acc)
print()
print("Precision :", prec)
print()
print("Recall :", rec)
print()
print("F1 :", f1)
print()

print(f"Mean Accuracy : {np.mean(acc):.4f}")
print(f"Std Accuracy  : {np.std(acc):.4f}")
print()

print(f"Mean Precision : {np.mean(prec):.4f}")
print(f"Mean Recall    : {np.mean(rec):.4f}")
print(f"Mean F1 Score  : {np.mean(f1):.4f}")

print("\n" + "=" * 60)
print(f"Best Fold : {best_fold}")
print(f"Best Accuracy : {best_fold_accuracy:.4f}")
print("Best model saved as:")
print("../models/best_mobilenetv3_screen_detector.pth")
print("=" * 60)


Cross Validation Results
Accuracy : [0.7619047619047619, 0.7142857142857143, 0.95, 0.65, 0.8]

Precision : [0.7777777777777778, 0.7272727272727273, 1.0, 0.6363636363636364, 0.875]

Recall : [0.7, 0.7272727272727273, 0.9, 0.7, 0.7]

F1 : [0.7368421052631579, 0.7272727272727273, 0.9473684210526315, 0.6666666666666666, 0.7777777777777778]

Mean Accuracy : 0.7752
Std Accuracy  : 0.1007

Mean Precision : 0.8033
Mean Recall    : 0.7455
Mean F1 Score  : 0.7712

Best Fold : 3
Best Accuracy : 0.9500
Best model saved as:
../models/best_mobilenetv3_screen_detector.pth


# Summary

This notebook evaluated MobileNetV3 Small using Stratified 5-Fold Cross Validation.

Despite its lightweight architecture and low inference cost, MobileNetV3 produced lower average performance than the deeper ResNet models.

The results establish an efficient baseline for comparison with the subsequent transfer learning experiments.